# 06 Export: build the files the Streamlit app needs
Retrain the Sprint 4 random forest, save it, and write trimmed movie and tag tables to app/.

- **Reads:** data/processed/movies.parquet, data/processed/tags.parquet
- **Writes:** app/model.joblib, app/data/movies_app.parquet, app/data/tags_app.parquet
- **Run after:** 02_clean_join and 05_recommend

What I found: the retrained forest scores the same test MAE as Sprint 4 (0.728); the model is 7.5 MB and the two data files about 5 MB.

In [1]:
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

PROCESSED = "../data/processed"
APP_DATA = "../app/data"
os.makedirs(APP_DATA, exist_ok=True)

movies = pd.read_parquet(f"{PROCESSED}/movies.parquet")
tags = pd.read_parquet(f"{PROCESSED}/tags.parquet")
print(movies.shape, tags.shape)

(49453, 34) (306637, 7)


In [2]:
movies = movies.sort_values(["year", "tconst"]).reset_index(drop=True)
by_director = movies.groupby("director_id")["rating"]
movies["dir_prev_films"] = by_director.cumcount()
movies["dir_prev_avg"] = by_director.transform(lambda s: s.shift(1).expanding().mean())

SPLIT_YEAR = 2015
train = movies[movies["year"] < SPLIT_YEAR]
genre_cols = [c for c in movies.columns if c.startswith("g_")]
runtime_fill = train["runtime"].median()
dir_fill = train["rating"].mean()

def make_features(df):
    X = pd.DataFrame(index=df.index)
    X["year"] = df["year"].astype(float)
    X["runtime"] = df["runtime"].astype(float).fillna(runtime_fill)
    X["dir_prev_films"] = df["dir_prev_films"].fillna(0)
    X["dir_prev_avg"] = df["dir_prev_avg"].fillna(dir_fill)
    X["has_dir_history"] = df["dir_prev_avg"].notna().astype(int)
    X[genre_cols] = df[genre_cols]
    return X

X_all = make_features(movies)
is_train = movies["year"] < SPLIT_YEAR

In [3]:
rf = RandomForestRegressor(n_estimators=200, min_samples_leaf=20, random_state=42, n_jobs=-1)
rf.fit(X_all[is_train], movies.loc[is_train, "rating"])

test_mae = mean_absolute_error(movies.loc[~is_train, "rating"], rf.predict(X_all[~is_train]))
print("Forest test MAE:", round(test_mae, 3))

Forest test MAE: 0.728


In [4]:
joblib.dump(rf, "../app/model.joblib", compress=3)
print("model size MB:", round(os.path.getsize("../app/model.joblib") / 1e6, 1))

model size MB: 7.5


In [5]:
app_movies = movies[["tconst", "title", "year", "runtime", "rating", "votes", "director", "genres"]].copy()
app_movies["decade"] = (app_movies["year"] // 10 * 10).astype("int16")
app_movies["split"] = np.where(is_train, "train", "test")
app_movies = pd.concat([app_movies, X_all.add_prefix("x_")], axis=1)
app_movies = app_movies.sort_values("votes", ascending=False).reset_index(drop=True)
app_movies.to_parquet(f"{APP_DATA}/movies_app.parquet", index=False)

app_tags = tags[["tconst", "kind", "tag", "label", "n_films", "weight"]]
app_tags.to_parquet(f"{APP_DATA}/tags_app.parquet", index=False)

for f in sorted(os.listdir(APP_DATA)):
    print(f, round(os.path.getsize(f"{APP_DATA}/{f}") / 1e6, 1), "MB")
print(app_movies.shape, app_tags.shape)

movies_app.parquet 2.3 MB
tags_app.parquet 2.6 MB
tv_seasons_app.parquet 0.0 MB
tv_sharks_app.parquet 0.0 MB
(49453, 39) (306637, 6)
